# Azure Data Lake Storage Gen2 — ABFS + OAuth (no DBFS mount)

This notebook configures **Spark** to read/write Azure Data Lake Storage Gen2 using the **ABFS** URI scheme (`abfss://...`) and **OAuth** via `spark.conf` (Service Principal). **We do not use `dbutils.fs.mount`** here.

---

## Prerequisites

1. ADLS Gen2 storage account and container with your data.
2. Service Principal with **Storage Blob Data Contributor** (or equivalent) on the container.
3. Fill in **tenant_id**, **client_id**, **client_secret** below (or use `dbutils.secrets.get` for production).

**Expected data layout** under the container (folder `data/`):
- `data/json/2015-summary.json`
- `data/2010-summary.csv`

After this runs, `base_path` points at `abfss://<container>@<account>.dfs.core.windows.net/data`.

---

## How lesson notebooks connect

**All hands-on notebooks that read ADLS** (`01-Day1-...`, `01-Day2-...`, `01-Day3-...`, `03-Day3-...`, `01-Day4-...`, `03-Day4-...`, `01-Day5-...`, `03-Day5-...`, `04-Day5-...`) must start with **`%run ./02-Mount-Azure-Data-Lake`** (typically the **first code cell**) so **`spark.conf` OAuth** and **`base_path`** are set on the cluster.

- Keep **`02-Mount-Azure-Data-Lake.ipynb`** in the **same workspace folder** as the lesson notebook, or adjust the path (e.g. `%run ../notebooks/02-Mount-Azure-Data-Lake`).
- After a **new cluster** or **RESTART**, run **`%run`** again, then the lesson **paths** cell (`BASE_PATH = base_path`, etc.).
- You can **Run all** on this notebook alone for debugging; in class it is normally executed **only** via `%run` from other notebooks.

## Step 1 — Storage account and OAuth (Service Principal)

In [ ]:
# Azure ADLS Gen2 storage account and container (your working paths)
adlsAccountName = "sadbtrng19032026"
containerName = "atininput"

# Service Principal
tenant_id = "2a259c7d-f11a-4d74-846e-900f4c90805a"
client_id = "7ee1d98c-66ac-473b-a2b7-149ab3decd3a"
client_secret = ""  # paste your client secret here (or use dbutils.secrets)


In [ ]:
# OAuth configuration on spark.conf (per storage account)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{adlsAccountName}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)
spark.conf.set(f"fs.azure.account.auth.type.{adlsAccountName}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{adlsAccountName}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{adlsAccountName}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{adlsAccountName}.dfs.core.windows.net",
    client_secret
)
print("OAuth configured for account:", adlsAccountName)

## Step 2 — Base path (ABFSS) and optional smoke read

In [ ]:
base_path = f"abfss://{containerName}@{adlsAccountName}.dfs.core.windows.net/data"
print("base_path:", base_path)

In [ ]:
# Optional: verify CSV read (same path pattern as Day 1 main notebook)
file_path = f"{base_path}/2010-summary.csv"
df_smoke = spark.read.format("csv").option("header", True).option("inferSchema", True).load(file_path)
df_smoke.show(5)